# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sarahnjunge/starter-notebooks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The starter dataset summarizes each page using historical search-performance windows: the previous 30 days, the latest 30 days, and a 90-day aggregate. There is no explicit observation date in the starter CSV, so I do not claim a calendar-date range.

For the starter task, the prediction unit is therefore one content page, using the available historical performance windows to identify pages that may require attention.

In [6]:
!git clone https://github.com/Sarahnjunge/starter-notebooks.git
%cd starter-notebooks

Cloning into 'starter-notebooks'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 158 (delta 64), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.88 MiB | 15.78 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/starter-notebooks


In [7]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded:", df.shape)

Dataset loaded: (30000, 44)


In [8]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nUnique content pages:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

print("\nTime-window fields:")
time_fields = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

display(df[time_fields].head())

Rows: 30000
Columns: 44

Unique content pages: 30000
Unique clients: 32

Time-window fields:


,impressions_90d,clicks_90d,sessions_90d,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d
0,3803,29,17,578,2,2,987,13,9
1,15320,7,9,2501,2,3,5915,1,2
2,12581,11,11,2382,1,1,6089,3,3
3,11751,58,78,3626,22,35,4206,17,26
4,19140,24,145,4211,10,14,6452,2,9


## 2. Fields: feature / label / context / excluded

**Features:** historical search, traffic, engagement, content, freshness, and ranking signals that are available before making the prediction. These include impressions, clicks, sessions, CTR, average position, engagement rate, content age, days since last update, search volume, competition, and content characteristics.

**Label:** `trend_direction`. For the starter dataset, this is the proxy target used to identify whether a content page is declining or not declining.

**Context:** `content_id` and `client_id`. These identify the page and client but are not used as predictive features.

**Excluded:** `trend_pct`, because it is used to derive `trend_direction` and would therefore directly reveal information about the proxy label. I also exclude `trend_direction` from the features because it is the target itself. Identifiers are excluded from model features because they identify observations rather than describe their behavior.

In [9]:
# Check the fields used in the data contract

features = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier"
]

label = "trend_direction"

context = [
    "content_id",
    "client_id"
]

excluded = [
    "trend_pct",
    "trend_direction"
]

print("Number of features:", len(features))
print("\nLabel:", label)
print("\nContext fields:", context)
print("\nExcluded fields:", excluded)

print("\nMissing feature values:")
display(df[features].isna().sum().sort_values(ascending=False).head(10))


Number of features: 38

Label: trend_direction

Context fields: ['content_id', 'client_id']

Excluded fields: ['trend_pct', 'trend_direction']

Missing feature values:


,0
char_count,7699
word_count,7699
word_count_tier,7699
char_count_tier,7699
competition_level,2610
search_volume,2468
cpc,2468
competition,2468
main_intent,2374
scroll_rate,125


## 3. Verify it with queries (grain, counts, missing values, windows)

I verify the contract using checks on row uniqueness, field availability, target distribution, missing values, and the historical performance windows.

The checks confirm that each row represents one content page, that the target is available for the starter dataset, and that the historical 30-day and 90-day fields are present. The missing-value check is also used to identify fields that may require preprocessing before modeling.

In [10]:
# 1. Verify row grain

print("Total rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

# 2. Verify target availability

print("\nTarget availability:")
print("Missing trend_direction:", df["trend_direction"].isna().sum())
print("Unique target values:", df["trend_direction"].nunique())

display(df["trend_direction"].value_counts(dropna=False))

# 3. Verify historical windows

print("\nHistorical window fields:")
window_fields = [
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_90d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_90d",
    "sessions_last_30d",
    "sessions_prev_30d"
]

display(df[window_fields].isna().sum().to_frame("missing_values"))

# 4. Verify missing values in planned features

print("\nFeatures with missing values:")
missing = df[features].isna().sum()
display(
    missing[missing > 0]
    .sort_values(ascending=False)
    .to_frame("missing_values")
)


Total rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0

Target availability:
Missing trend_direction: 0
Unique target values: 5


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



Historical window fields:


,missing_values
impressions_90d,0
impressions_last_30d,0
impressions_prev_30d,0
clicks_90d,0
clicks_last_30d,0
clicks_prev_30d,0
sessions_90d,0
sessions_last_30d,0
sessions_prev_30d,0



Features with missing values:


,missing_values
char_count_tier,7699
word_count,7699
word_count_tier,7699
char_count,7699
competition_level,2610
search_volume,2468
competition,2468
cpc,2468
main_intent,2374
scroll_rate,125


## 4. Data limits

The starter dataset has several important limits.

First, it does not contain an explicit calendar date, so I cannot establish a real-world training or prediction date from this file.

Second, the target `trend_direction` is a proxy label derived from `trend_pct`, rather than a directly observed future outcome. This means the starter dataset can show patterns associated with the defined trend categories, but it cannot prove that a page will decline in the future.

Third, the target is imbalanced: the `down` class contains 16,262 pages, while the other categories have fewer observations. This makes accuracy alone a poor measure of whether the model is useful.

Finally, several features contain missing values. In particular, `word_count` and `char_count` are missing for 7,699 pages. These missing values need to be handled during preprocessing rather than silently dropped.

The starter dataset also cannot establish causality. A relationship between a feature and `trend_direction` does not prove that changing that feature would cause a page's performance to change.

In [11]:

print("Target distribution:")
target_counts = df["trend_direction"].value_counts()
target_pct = df["trend_direction"].value_counts(normalize=True) * 100

display(
    pd.DataFrame({
        "count": target_counts,
        "percentage": target_pct.round(2)
    })
)

print("\nMissing values in features:")
missing_features = df[features].isna().sum()
display(
    missing_features[missing_features > 0]
    .sort_values(ascending=False)
    .to_frame("missing_values")
)

print("\nDate fields in dataset:")
date_like_columns = [
    col for col in df.columns
    if "date" in col.lower() or "time" in col.lower()
]

print(date_like_columns)


Target distribution:


,count,percentage
trend_direction,,
down,16262,54.21
stable,5962,19.87
up,4388,14.63
new,2236,7.45
flat,1152,3.84



Missing values in features:


,missing_values
char_count_tier,7699
word_count,7699
word_count_tier,7699
char_count,7699
competition_level,2610
search_volume,2468
competition,2468
cpc,2468
main_intent,2374
scroll_rate,125



Date fields in dataset:
['days_since_last_update']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.